# LLM Judge API Host

This notebook runs GPT-OSS-20B on Colab and exposes it as an API endpoint you can call from your local machine.

**Usage:**
1. Run all cells
2. Copy the ngrok URL printed at the end
3. Set `LLM_JUDGE_URL=<ngrok_url>` in your local environment
4. Call the API with POST requests

## 1. Setup Environment

In [ ]:
!pip install -q --upgrade torch transformers triton==3.4 kernels flask flask-cors pyngrok

In [ ]:
!pip uninstall -q torchvision torchaudio -y

**⚠️ IMPORTANT:** Restart your Colab runtime now (Runtime > Restart session), then continue from cell 3.

## 2. Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "openai/gpt-oss-20b"

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="cuda",
)
print("✅ Model loaded successfully!")

## 3. Define Generation Function

In [ ]:
def generate_response(messages, max_tokens=2048, temperature=0.2):
    """Generate a response from the model."""
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
    )
    
    input_len = inputs["input_ids"].shape[-1]
    output_ids = generated_ids[0][input_len:]
    response_text = tokenizer.decode(output_ids, skip_special_tokens=True)
    
    return response_text

# Quick test
test_response = generate_response([{"role": "user", "content": "Say hello!"}], max_tokens=50)
print(f"Test response: {test_response}")

## 4. Create Flask API Server

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading

app = Flask(__name__)
CORS(app)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok", "model": MODEL_ID})

@app.route('/v1/chat/completions', methods=['POST'])
def chat_completions():
    """OpenAI-compatible chat completions endpoint."""
    try:
        data = request.json
        messages = data.get('messages', [])
        max_tokens = data.get('max_tokens', 2048)
        temperature = data.get('temperature', 0.2)
        
        print(f"📥 Received request: {len(messages)} messages, max_tokens={max_tokens}")
        
        response_text = generate_response(messages, max_tokens, temperature)
        
        print(f"📤 Generated {len(response_text)} chars")
        
        # Return OpenAI-compatible response format
        return jsonify({
            "id": "colab-judge",
            "object": "chat.completion",
            "model": MODEL_ID,
            "choices": [{
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": response_text
                },
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": len(str(messages)) // 4,
                "completion_tokens": len(response_text) // 4,
                "total_tokens": (len(str(messages)) + len(response_text)) // 4
            }
        })
    except Exception as e:
        print(f"❌ Error: {e}")
        return jsonify({"error": str(e)}), 500

@app.route('/evaluate', methods=['POST'])
def evaluate():
    """Custom endpoint for tutorial evaluation."""
    try:
        data = request.json
        content_a = data.get('content_a', '')
        content_b = data.get('content_b', '')
        codebase_context = data.get('codebase_context', '')
        
        prompt = f"""You are a meticulous technical documentation judge conducting a blind A/B test.
Your goal: Determine which Tutorial Series better teaches developers.

## Codebase Context
{codebase_context}

## Tutorial Series A
{content_a[:6000]}

## Tutorial Series B
{content_b[:6000]}

---

Score each on FIDELITY (code accuracy), PEDAGOGY (teaching quality), COVERAGE (completeness) from 1-5.

Return ONLY valid JSON:
{{
    "winner": "A" or "B" or "Tie",
    "fidelity_A": <1-5>,
    "fidelity_B": <1-5>,
    "pedagogy_A": <1-5>,
    "pedagogy_B": <1-5>,
    "coverage_A": <1-5>,
    "coverage_B": <1-5>,
    "rationale": "Brief explanation..."
}}
"""
        
        response_text = generate_response(
            [{"role": "user", "content": prompt}],
            max_tokens=1500,
            temperature=0.1
        )
        
        return jsonify({"response": response_text})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

print("✅ Flask app defined")

## 5. Start Server with ngrok

In [ ]:
from pyngrok import ngrok
import threading

# Optional: Set your ngrok auth token for longer sessions
# Get your token from https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

PORT = 5000

# Start Flask in a background thread
def run_flask():
    app.run(host='0.0.0.0', port=PORT, use_reloader=False)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

# Create ngrok tunnel
public_url = ngrok.connect(PORT)

print("\n" + "="*60)
print("🚀 LLM JUDGE API IS RUNNING!")
print("="*60)
print(f"\n📡 Public URL: {public_url}")
print(f"\n🔧 Set this in your local .env:")
print(f"   LLM_JUDGE_URL={public_url}")
print("\n📚 Endpoints:")
print(f"   GET  {public_url}/health")
print(f"   POST {public_url}/v1/chat/completions  (OpenAI-compatible)")
print(f"   POST {public_url}/evaluate  (Custom evaluation endpoint)")
print("\n" + "="*60)

## 6. Test the API

In [ ]:
import requests

# Test health endpoint
url = str(public_url)
response = requests.get(f"{url}/health")
print(f"Health check: {response.json()}")

# Test chat completions
response = requests.post(
    f"{url}/v1/chat/completions",
    json={
        "messages": [{"role": "user", "content": "What is 2+2?"}],
        "max_tokens": 100
    }
)
print(f"\nChat test: {response.json()['choices'][0]['message']['content']}")

## 7. Keep Alive

Run this cell to keep the notebook active. The API will stay running as long as this cell is executing.

In [ ]:
import time

print("🔄 Keeping server alive... (Interrupt to stop)")
print(f"📡 API URL: {public_url}")

try:
    while True:
        time.sleep(60)
        print(f"💓 Server still running at {public_url}")
except KeyboardInterrupt:
    print("\n🛑 Server stopped")
    ngrok.disconnect(public_url)